<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Kokoro-82M_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Kokoro-82M — 82M-param Apache-2.0 TTS

Kokoro is an 82-million-parameter TTS model that punches well above its weight. It runs comfortably on **CPU** (no GPU required) and only takes ~330 MB of disk, yet delivers audio quality comparable to models 10× its size. Apache-2.0 licensed, 54 voices across 9 languages, and already deployed in production at major TTS APIs.

> *As of April 2025 the market rate of Kokoro served over API is under $1 per million characters of text input.* — [`hexgrad/Kokoro-82M`](https://huggingface.co/hexgrad/Kokoro-82M)

**Upstream**: [hexgrad/Kokoro-82M](https://huggingface.co/hexgrad/Kokoro-82M) · [hexgrad/kokoro](https://github.com/hexgrad/kokoro) · [arXiv 2306.07691](https://arxiv.org/abs/2306.07691) (StyleTTS 2) · [arXiv 2203.02395](https://arxiv.org/abs/2203.02395) (iSTFTNet) · [Apache-2.0](https://huggingface.co/hexgrad/Kokoro-82M/blob/main/LICENSE)

## What this notebook does

- **Run Kokoro-82M** locally in Colab — GPU is optional; CPU works fine for short texts.
- **54 voices across 9 languages** (English 🇺🇸🇬🇧, Spanish 🇪🇸, French 🇫🇷, Hindi 🇮🇳, Italian 🇮🇹, Brazilian Portuguese 🇧🇷, Japanese 🇯🇵, Mandarin 🇨🇳).
- **Custom pronunciation** via Markdown-link syntax: `[word](/IPA/)` → e.g. `[Kokoro](/kˈOkəɹO/)`.
- **Voice blending**: pass `af_bella,am_michael` to average two voices together.
- **Streaming output** for long texts (chunks via `\n+` or auto).
- **Tokenize-only mode** to inspect G2P phonemes without generating audio.
- **Cache to Google Drive** so weights survive Colab restarts.

## What this notebook does NOT do

- Voice cloning from a reference audio — Kokoro has only built-in voice packs, not a TTS-with-cloning pipeline.
- Training or fine-tuning. This is an inference-only notebook.
- Real-time/low-latency streaming over a network (browser audio only, not WebRTC).

## How to use

1. Run **STEP 1** (installs `kokoro`, `soundfile`, `espeak-ng`).
2. *Optional* — run **STEP 2** to pre-cache the model + voice packs to Drive.
3. Run **STEP 3** to load the core functions.
4. Run **STEP 4** for the interactive Gradio UI.
5. *Optional* — run **STEP 5** to prevent Colab from disconnecting during long sessions.
6. *Optional* — run **STEP 6** for a single one-off generation, or **STEP 7** for batch synthesis.

> **Tip:** First-time model load takes ~3–10 s. Subsequent generations are <1 s per short sentence.

## Citation

```bibtex
@inproceedings{kokoro-82m,
  title={{Kokoro-82M}: An Open-Weight 82M-Param TTS Model},
  author={{hexgrad}},
  year={{2025}},
  url={{https://huggingface.co/hexgrad/Kokoro-82M}}
}}
```


In [ ]:
# @title STEP 1 — Install Dependencies
# @markdown Installs the `kokoro` PyPI package, `soundfile`, `numpy`, and the `espeak-ng` system package required by `misaki`. Optional extras for Japanese (`misaki[ja]`) and Chinese (`misaki[zh]`) are listed at the bottom.

import os, sys, subprocess, importlib, time

# GPU auto-detect (Kokoro is CPU-first; GPU is optional and only marginally faster on small models)
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'[GPU] torch {torch.__version__} · device = {DEVICE}', flush=True)
if DEVICE == 'cuda':
    try:
        name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'[GPU] {name} ({vram:.1f} GB VRAM)', flush=True)
    except Exception as e:
        print(f'[GPU] (could not read device name: {e})', flush=True)

# Drive cache setup — Kokoro weights are tiny (~330 MB) so Drive caching is optional but recommended
DRIVE = '/content/drive/MyDrive/AEI_TTS_Cache/huggingface'
os.environ.setdefault('HF_HOME', DRIVE)
os.makedirs(DRIVE, exist_ok=True)
print(f'[Cache] HF_HOME = {DRIVE}', flush=True)

# Output dirs
os.makedirs('/content/audio_out', exist_ok=True)
print('[Setup] /content/audio_out ready.', flush=True)

# Install system package: espeak-ng (required by misaki[en] for OOD-word fallback)
print('\n[apt] Installing espeak-ng...', flush=True)
subprocess.run(['apt-get', '-qq', '-y', 'install', 'espeak-ng'],
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
import shutil
espeak = shutil.which('espeak-ng')
print(f'[apt] espeak-ng = {espeak}', flush=True)

# Install pip packages (silent)
t0 = time.time()
print('\n[pip] Installing kokoro, soundfile, numpy, tqdm, gradio...', flush=True)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'kokoro>=0.9.4',
     'soundfile', 'numpy', 'tqdm', 'hf_transfer', 'gradio>=4.0'],
    check=True,
)
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
print(f'[pip] Done in {time.time()-t0:.1f}s', flush=True)

# Verify Kokoro import works
try:
    import kokoro
    print(f'[kokoro] version = {kokoro.__version__}', flush=True)
except Exception as e:
    print(f'[kokoro] import FAILED: {e}', flush=True)
    raise

# Check optional language packs (Japanese / Chinese). Install on demand.
for pkg, lang in [('misaki[ja]', 'Japanese'), ('misaki[zh]', 'Mandarin')]:
    try:
        importlib.import_module('misaki.ja' if lang == 'Japanese' else 'misaki.zh')
        print(f'[misaki] {lang} pack present', flush=True)
    except ImportError:
        print(f'[misaki] {lang} pack MISSING — pip install misaki[{lang[:2]}] if needed', flush=True)

print('\n[Done] STEP 1 complete. Move on to STEP 2 (pre-cache) or STEP 3 (core).', flush=True)


In [ ]:
# @title STEP 2 — Pre-cache Models and Sample Assets
# @markdown Downloads the Kokoro-82M model + 54 voice packs to the local cache (or your Drive if mounted). Also fetches `en.txt` (random quote pool), `gatsby5k.md`, and `frankenstein5k.md` from the upstream Space for the fill buttons in the UI.

# @markdown **Kokoro model size: ~330 MB**. 54 voice packs: ~75 MB total. Skips download if already cached.

import os
from huggingface_hub import snapshot_download
import urllib.request

# Kokoro model repo
KOKORO_REPO = 'hexgrad/Kokoro-82M'
print(f'[Download] {KOKORO_REPO} ...')
snapshot_download(KOKORO_REPO)
print(f'[Download] {KOKORO_REPO} cached.')

# Sample assets (from hexgrad/Kokoro-TTS Space, used by the fill buttons)
SAMPLE_DIR = '/content/kokoro_samples'
os.makedirs(SAMPLE_DIR, exist_ok=True)
SAMPLES = {
    'en.txt':             'https://huggingface.co/spaces/hexgrad/Kokoro-TTS/resolve/main/en.txt',
    'gatsby5k.md':        'https://huggingface.co/spaces/hexgrad/Kokoro-TTS/resolve/main/gatsby5k.md',
    'frankenstein5k.md':  'https://huggingface.co/spaces/hexgrad/Kokoro-TTS/resolve/main/frankenstein5k.md',
}
print('\n[Download] Sample text assets (en.txt + books)...')
for name, url in SAMPLES.items():
    dst = os.path.join(SAMPLE_DIR, name)
    if os.path.exists(dst) and os.path.getsize(dst) > 100:
        print(f'  ✓ {name} ({os.path.getsize(dst)/1024:.1f} KB) — cached')
        continue
    try:
        urllib.request.urlretrieve(url, dst)
        print(f'  ✓ {name} ({os.path.getsize(dst)/1024:.1f} KB)')
    except Exception as e:
        print(f'  ✗ {name} — {e} (will fall back to placeholder text)')

print('\n[Done] STEP 2 complete. All weights + sample assets cached.')


In [ ]:
# @title STEP 3 — Core Functions (lazy model loading)
# @markdown Defines the inference wrappers, voice catalog, and helpers. Models are loaded on first use per language, not at import time. This cell is fast.

import os, time, gc, json
import numpy as np
import torch
import soundfile as sf
from typing import Optional, List, Tuple, Generator

# ── Voice catalog (54 voices across 9 languages) ───────────────────────
# (id, lang_code, display, traits, quality, duration, grade)
VOICES = [('af_heart', 'a', 'Heart (🚺❤️)', '❤️ warm, premium', 'A', '10+ hrs', 'A'), ('af_alloy', 'a', 'Alloy', 'neutral female', 'B', 'MM min', 'C'), ('af_aoede', 'a', 'Aoede', 'neutral female', 'B', 'H hrs', 'C+'), ('af_bella', 'a', 'Bella (🔥)', '🔥 expressive, premium', 'A', 'HH hrs', 'A-'), ('af_jessica', 'a', 'Jessica', 'neutral female', 'C', 'MM min', 'D'), ('af_kore', 'a', 'Kore', 'neutral female', 'B', 'H hrs', 'C+'), ('af_nicole', 'a', 'Nicole (🎧)', '🎧 ASMR-like', 'B', 'HH hrs', 'B-'), ('af_nova', 'a', 'Nova', 'neutral female', 'B', 'MM min', 'C'), ('af_river', 'a', 'River', 'neutral female', 'C', 'MM min', 'D'), ('af_sarah', 'a', 'Sarah', 'neutral female', 'B', 'H hrs', 'C+'), ('af_sky', 'a', 'Sky', 'neutral female', 'B', 'M min 🤏', 'C-'), ('am_adam', 'a', 'Adam', 'neutral male', 'D', 'H hrs', 'F+'), ('am_echo', 'a', 'Echo', 'neutral male', 'C', 'MM min', 'D'), ('am_eric', 'a', 'Eric', 'neutral male', 'C', 'MM min', 'D'), ('am_fenrir', 'a', 'Fenrir', 'neutral male', 'B', 'H hrs', 'C+'), ('am_liam', 'a', 'Liam', 'neutral male', 'C', 'MM min', 'D'), ('am_michael', 'a', 'Michael', 'neutral male', 'B', 'H hrs', 'C+'), ('am_onyx', 'a', 'Onyx', 'neutral male', 'C', 'MM min', 'D'), ('am_puck', 'a', 'Puck', 'neutral male', 'B', 'H hrs', 'C+'), ('am_santa', 'a', 'Santa', 'neutral male', 'C', 'M min 🤏', 'D-'), ('bf_alice', 'b', 'Alice (UK)', 'neutral female', 'C', 'MM min', 'D'), ('bf_emma', 'b', 'Emma (UK)', 'neutral female', 'B', 'HH hrs', 'B-'), ('bf_isabella', 'b', 'Isabella (UK)', 'neutral female', 'B', 'MM min', 'C'), ('bf_lily', 'b', 'Lily (UK)', 'neutral female', 'C', 'MM min', 'D'), ('bm_daniel', 'b', 'Daniel (UK)', 'neutral male', 'C', 'MM min', 'D'), ('bm_fable', 'b', 'Fable (UK)', 'neutral male', 'B', 'MM min', 'C'), ('bm_george', 'b', 'George (UK)', 'neutral male', 'B', 'MM min', 'C'), ('bm_lewis', 'b', 'Lewis (UK)', 'neutral male', 'C', 'H hrs', 'D+'), ('jf_alpha', 'j', 'Alpha (日本)', 'neutral female', 'B', 'H hrs', 'C+'), ('jf_gongitsune', 'j', 'Gongitsune (日本)', 'CC BY', 'B', 'MM min', 'C'), ('jf_nezumi', 'j', 'Nezumi (日本)', 'CC BY', 'B', 'M min 🤏', 'C-'), ('jf_tebukuro', 'j', 'Tebukuro (日本)', 'CC BY', 'B', 'MM min', 'C'), ('jm_kumo', 'j', 'Kumo (日本)', 'CC BY', 'B', 'M min 🤏', 'C-'), ('zf_xiaobei', 'z', 'Xiaobei (中文)', 'neutral female', 'C', 'MM min', 'D'), ('zf_xiaoni', 'z', 'Xiaoni (中文)', 'neutral female', 'C', 'MM min', 'D'), ('zf_xiaoxiao', 'z', 'Xiaoxiao (中文)', 'neutral female', 'C', 'MM min', 'D'), ('zf_xiaoyi', 'z', 'Xiaoyi (中文)', 'neutral female', 'C', 'MM min', 'D'), ('zm_yunjian', 'z', 'Yunjian (中文)', 'neutral male', 'C', 'MM min', 'D'), ('zm_yunxi', 'z', 'Yunxi (中文)', 'neutral male', 'C', 'MM min', 'D'), ('zm_yunxia', 'z', 'Yunxia (中文)', 'neutral male', 'C', 'MM min', 'D'), ('zm_yunyang', 'z', 'Yunyang (中文)', 'neutral male', 'C', 'MM min', 'D'), ('ef_dora', 'e', 'Dora (es)', 'neutral female', '—', '—', '—'), ('em_alex', 'e', 'Alex (es)', 'neutral male', '—', '—', '—'), ('em_santa', 'e', 'Santa (es)', 'neutral male', '—', '—', '—'), ('ff_siwis', 'f', 'Siwis (fr)', 'CC BY SIWIS', 'B', '<11 h', 'B-'), ('hf_alpha', 'h', 'Alpha (हिं)', 'neutral female', 'B', 'MM min', 'C'), ('hf_beta', 'h', 'Beta (हिं)', 'neutral female', 'B', 'MM min', 'C'), ('hm_omega', 'h', 'Omega (हिं)', 'neutral male', 'B', 'MM min', 'C'), ('hm_psi', 'h', 'Psi (हिं)', 'neutral male', 'B', 'MM min', 'C'), ('if_sara', 'i', 'Sara (it)', 'neutral female', 'B', 'MM min', 'C'), ('im_nicola', 'i', 'Nicola (it)', 'neutral male', 'B', 'MM min', 'C'), ('pf_dora', 'p', 'Dora (pt-BR)', 'neutral female', '—', '—', '—'), ('pm_alex', 'p', 'Alex (pt-BR)', 'neutral male', '—', '—', '—'), ('pm_santa', 'p', 'Santa (pt-BR)', 'neutral male', '—', '—', '—')]

LANG_TABLE = [('a', 'American English 🇺🇸', 'Built-in (misaki[en])'), ('b', 'British English 🇬🇧', 'Built-in (misaki[en])'), ('e', 'Spanish 🇪🇸', 'espeak-ng fallback'), ('f', 'French 🇫🇷', 'espeak-ng fallback'), ('h', 'Hindi 🇮🇳', 'espeak-ng fallback'), ('i', 'Italian 🇮🇹', 'espeak-ng fallback'), ('p', 'Brazilian Portuguese 🇧🇷', 'espeak-ng fallback'), ('j', 'Japanese 🇯🇵', 'Requires: pip install misaki[ja]'), ('z', 'Mandarin Chinese 🇨🇳', 'Requires: pip install misaki[zh]')]

# Index voices by language for the UI dropdown
VOICES_BY_LANG = {}
for v in VOICES:
    VOICES_BY_LANG.setdefault(v[1], []).append(v)

def list_voices(lang_code: str = None) -> list:
    """Return voice list. If lang_code is None, return all 54."""
    if lang_code and lang_code in VOICES_BY_LANG:
        return VOICES_BY_LANG[lang_code]
    return VOICES

# ── Lazy KPipeline cache (one per language) ─────────────────────────────
PIPELINES = {}  # lang_code -> KPipeline
SAMPLE_RATE = 24000  # Kokoro's fixed output sample rate

def get_pipeline(lang_code: str = 'a', device: str = None):
    """Lazily build a KPipeline for the given language.
    lang_code: 'a' (US) | 'b' (UK) | 'e' (es) | 'f' (fr) | 'h' (hi)
               | 'i' (it) | 'p' (pt-BR) | 'j' (ja) | 'z' (zh)
    device: 'cuda' | 'cpu' | None (auto)
    """
    if lang_code in PIPELINES and PIPELINES[lang_code] is not None:
        return PIPELINES[lang_code]
    if device is None:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
    from kokoro import KPipeline
    print(f'[KPipeline] lang_code={lang_code!r} device={device!r} ...', flush=True)
    t0 = time.time()
    pipe = KPipeline(lang_code=lang_code, device=device, repo_id="hexgrad/Kokoro-82M")
    PIPELINES[lang_code] = pipe
    print(f'[KPipeline] ready in {time.time()-t0:.1f}s', flush=True)
    return pipe

def get_tokenizer(lang_code: str = 'a'):
    """Return a phoneme-only pipeline (no model loaded, no audio generated)."""
    from kokoro import KPipeline
    return KPipeline(lang_code=lang_code, model=False, repo_id="hexgrad/Kokoro-82M")

def free_pipeline(lang_code: str = None):
    """Release one or all pipelines from memory."""
    global PIPELINES
    if lang_code is None:
        for k in list(PIPELINES.keys()):
            free_pipeline(k)
        return
    if lang_code in PIPELINES and PIPELINES[lang_code] is not None:
        del PIPELINES[lang_code]
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f'[Free] {lang_code!r} pipeline released', flush=True)

# ── Save helpers ─────────────────────────────────────────────────────────
def save_wav(audio: np.ndarray, sr: int, name: str) -> str:
    """Save a 1-D float numpy array as a 16-bit PCM WAV."""
    path = os.path.join('/content/audio_out', name)
    sf.write(path, audio, sr, subtype='PCM_16')
    return path

def save_wav_chunks(chunks: list, sr: int, name: str) -> str:
    """Concatenate and save a list of audio chunks as one WAV."""
    if not chunks:
        raise ValueError('No audio chunks to save.')
    return save_wav(np.concatenate(chunks), sr, name)

# ── Synthesis wrappers ───────────────────────────────────────────────────
def synthesize(text: str,
               voice: str = 'af_heart',
               speed: float = 1.0,
               lang_code: str = 'a',
               split_pattern: Optional[str] = r'\n+',
               device: str = None
              ) -> List[Tuple[str, str, np.ndarray]]:
    """Generate audio chunks. Returns a list of (graphemes, phonemes, audio_array) tuples.
    Kokoro chunks long text automatically (default split on \n+)."""
    if not text or not text.strip():
        raise ValueError('Text is empty.')
    pipe = get_pipeline(lang_code, device=device)
    results = []
    for i, result in enumerate(pipe(text, voice=voice, speed=speed, split_pattern=split_pattern)):
        if result.audio is None:
            continue
        audio = result.audio.numpy() if isinstance(result.audio, torch.Tensor) else np.asarray(result.audio)
        results.append((result.graphemes, result.phonemes, audio))
    return results

def synthesize_concat(text: str,
                     voice: str = 'af_heart',
                     speed: float = 1.0,
                     lang_code: str = 'a',
                     split_pattern: Optional[str] = r'\n+',
                     device: str = None
                    ) -> Tuple[np.ndarray, int]:
    """Generate audio for the whole text and concatenate chunks into one array."""
    chunks = synthesize(text, voice, speed, lang_code, split_pattern, device=device)
    if not chunks:
        raise RuntimeError('No audio generated.')
    return np.concatenate([c[2] for c in chunks]), SAMPLE_RATE

def synthesize_streaming(text: str,
                        voice: str = 'af_heart',
                        speed: float = 1.0,
                        lang_code: str = 'a',
                        split_pattern: Optional[str] = r'\n+',
                        device: str = None
                       ) -> Generator[Tuple[str, str, np.ndarray], None, None]:
    """Streaming variant: yields (graphemes, phonemes, audio_array) as each chunk is generated."""
    if not text or not text.strip():
        return
    pipe = get_pipeline(lang_code, device=device)
    for result in pipe(text, voice=voice, speed=speed, split_pattern=split_pattern):
        if result.audio is None:
            continue
        audio = result.audio.numpy() if isinstance(result.audio, torch.Tensor) else np.asarray(result.audio)
        yield (result.graphemes, result.phonemes, audio)

def tokenize_only(text: str, lang_code: str = 'a') -> List[Tuple[str, str]]:
    """Return list of (graphemes, phonemes) without generating audio. Fast."""
    pipe = get_tokenizer(lang_code)
    out = []
    for result in pipe(text, voice='af_heart'):  # voice is required by API but ignored in tokenize mode
        out.append((result.graphemes, result.phonemes))
    return out

# ── Reproducibility ──────────────────────────────────────────────────────
def set_seed(seed: int = 42):
    """Seed torch for any stochastic ops. Kokoro inference is mostly deterministic
    for a given voice + speed, but seeding is good hygiene."""
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

# ── Sample text loaders (for the UI fill buttons) ────────────────────────
SAMPLE_DIR = '/content/kokoro_samples'

def _load_sample(name: str, fallback: str) -> str:
    path = os.path.join(SAMPLE_DIR, name)
    if os.path.exists(path) and os.path.getsize(path) > 100:
        with open(path, 'r', encoding='utf-8') as f:
            return f.read()
    return fallback

def random_quote() -> str:
    """Pick one random line from en.txt."""
    text = _load_sample('en.txt', '"The only way to do great work is to love what you do." — Steve Jobs')
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines:
        return text
    return __import__('random').choice(lines)

def gatsby_text() -> str:
    return _load_sample('gatsby5k.md', 'In my younger and more vulnerable years my father gave me some advice...')

def frankenstein_text() -> str:
    return _load_sample('frankenstein5k.md', 'You will rejoice to hear that no disaster has accompanied the commencement of an enterprise...')

print('[Core] Ready. Use STEP 4 for the UI, STEP 6 for quick test, STEP 7 for batch.')
print(f'[Core] {len(VOICES)} voices across {len(VOICES_BY_LANG)} languages.')


In [ ]:
# @title STEP 4 — Gradio UI
# @markdown Interactive web app with six tabs: **Generate** (default), **Stream**, **Tokens** (G2P-only), **Batch** (zip of all lines), **VRAM** (free loaded models), **Help** (pronunciation + voice codes).

import os, sys, time, json, io, zipfile
import numpy as np
import soundfile as sf
import gradio as gr

# ── Colab / nest_asyncio so Gradio can run inside the notebook loop ──
try:
    import nest_asyncio
    nest_asyncio.apply()
except Exception:
    pass
try:
    from IPython.display import clear_output as _clear
    _clear()
except Exception:
    pass

# ── Voice / language helpers ──────────────────────────────────────────
LANG_LABELS = [t[1] for t in LANG_TABLE]
LANG_CODES_BY_LABEL = {t[1]: t[0] for t in LANG_TABLE}

def voice_choices_for(lang_label: str):
    code = LANG_CODES_BY_LABEL.get(lang_label, 'a')
    return [(f"{v[2]} — {v[3]} [{v[5]}, {v[6]}]", v[0]) for v in VOICES_BY_LANG.get(code, [])]

def _normalize_voice(voice: str) -> str:
    """Gradio can pass either the raw voice id or a tuple; pick the id."""
    if isinstance(voice, (list, tuple)):
        return voice[-1] if voice else 'af_heart'
    if not voice:
        return 'af_heart'
    return voice

# ── Tab 1: Generate ───────────────────────────────────────────────────
def ui_generate(text, voice, speed, lang_label, progress=gr.Progress()):
    if not text or not text.strip():
        raise gr.Error('Text is empty.')
    voice = _normalize_voice(voice)
    code = LANG_CODES_BY_LABEL.get(lang_label, 'a')
    progress(0.0, 'Loading pipeline...')
    t0 = time.time()
    chunks = synthesize(text, voice, float(speed), code)
    if not chunks:
        raise gr.Error('No audio was generated.')
    audio, sr = np.concatenate([c[2] for c in chunks]), SAMPLE_RATE
    fname = f'generate_{int(time.time())}.wav'
    path = save_wav(audio, sr, fname)
    meta = (
        f"voice = {voice} · lang = {lang_label} · speed = {speed:.2f}\n"
        f"chunks = {len(chunks)} · duration = {len(audio)/sr:.2f}s · {time.time()-t0:.1f}s wall\n\n"
        + "\n\n".join(f"[{i}] {gs[:60]}{'…' if len(gs)>60 else ''}  →  {ps[:60]}{'…' if len(ps)>60 else ''}"
                        for i, (gs, ps, _) in enumerate(chunks))
    )
    return (sr, audio), path, meta

with gr.Blocks(title="Kokoro-82M", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Kokoro-82M — Apache-2.0 TTS\n82M params · 54 voices · 9 languages · CPU-friendly")

    with gr.Tabs():
        # ── Tab 1: Generate ──
        with gr.Tab("Generate"):
            with gr.Row():
                with gr.Column():
                    lang_dd = gr.Dropdown(choices=LANG_LABELS, value="American English 🇺🇸",
                                           label="Language",
                                           info="Picks the G2P backend and the voice list. Japanese and Chinese require `misaki[ja]` or `misaki[zh]`.")
                    voice_dd = gr.Dropdown(choices=voice_choices_for("American English 🇺🇸"),
                                            value="af_heart", label="Voice",
                                            info="Built-in voice pack. See VOICES.md for quality/duration grades.")
                    speed_sl = gr.Slider(0.5, 2.0, value=1.0, step=0.05, label="Speed",
                                          info="1.0 = normal. <1 = slower, >1 = faster.")
                    text_in = gr.Textbox(lines=6, label="Text",
                                          value="Hello! This is Kokoro-82M, an open-weight text-to-speech model with 82 million parameters.",
                                          info="Use Markdown-link syntax for custom pronunciation: [Kokoro](/kˈOkəɹO/)")
                    with gr.Row():
                        btn_rand = gr.Button("🎲 Random Quote", variant="secondary", scale=1)
                        btn_gatsby = gr.Button("📖 Gatsby", variant="secondary", scale=1)
                        btn_frank = gr.Button("🧪 Frankenstein", variant="secondary", scale=1)
                    btn_go = gr.Button("🔊 Generate", variant="primary")
                with gr.Column():
                    audio_out = gr.Audio(label="Audio", type="numpy",
                                          info="Click play to listen. Use the download icon to save.")
                    file_out = gr.File(label="Download .wav",
                                        info="Saved to /content/audio_out/.")
                    meta_out = gr.Textbox(label="Run metadata", lines=8, interactive=False)

            btn_rand.click(random_quote, outputs=text_in)
            btn_gatsby.click(gatsby_text, outputs=text_in)
            btn_frank.click(frankenstein_text, outputs=text_in)
            lang_dd.change(voice_choices_for, inputs=lang_dd, outputs=voice_dd)
            btn_go.click(ui_generate, inputs=[text_in, voice_dd, speed_sl, lang_dd],
                          outputs=[audio_out, file_out, meta_out])

        # ── Tab 2: Stream ──
        with gr.Tab("Stream"):
            gr.Markdown("Same as Generate, but chunks play back as they're produced. Best for long texts split by blank lines.")
            with gr.Row():
                with gr.Column():
                    s_lang = gr.Dropdown(choices=LANG_LABELS, value="American English 🇺🇸", label="Language",
                                          info="Language code passed to the G2P pipeline.")
                    s_voice = gr.Dropdown(choices=voice_choices_for("American English 🇺🇸"),
                                           value="af_heart", label="Voice",
                                           info="Voice pack. Streaming plays each chunk as it is generated.")
                    s_speed = gr.Slider(0.5, 2.0, value=1.0, step=0.05, label="Speed",
                                         info="Speech rate multiplier.")
                    s_text = gr.Textbox(lines=8, label="Text (split on \n\n for long-form)",
                                         info="Long texts are auto-chunked at sentence boundaries by the Kokoro pipeline.")
                    s_btn = gr.Button("▶ Stream", variant="primary")
                with gr.Column():
                    s_audio = gr.Audio(label="Streaming audio", type="numpy", autoplay=True,
                                        info="Chunks are concatenated and replayed in order. Each chunk is also saved individually to /content/audio_out/stream/.")
                    s_log = gr.Textbox(label="Chunk log", lines=10, interactive=False,
                                        info="One line per chunk, in order.")

            def ui_stream(text, voice, speed, lang_label, progress=gr.Progress()):
                if not text or not text.strip():
                    raise gr.Error('Text is empty.')
                voice = _normalize_voice(voice)
                code = LANG_CODES_BY_LABEL.get(lang_label, 'a')
                out_dir = '/content/audio_out/stream'
                os.makedirs(out_dir, exist_ok=True)
                pieces, log = [], []
                progress(0.0, 'Loading...')
                for i, (gs, ps, audio) in enumerate(synthesize_streaming(text, voice, float(speed), code)):
                    pieces.append(audio)
                    save_wav(audio, SAMPLE_RATE, os.path.join(out_dir, f'chunk_{i:03d}.wav'))
                    log.append(f"[{i}] {gs[:50]}{'…' if len(gs)>50 else ''}  →  {ps[:50]}{'…' if len(ps)>50 else ''}")
                    progress((i + 1) / 100.0, f"chunk {i+1}")
                if not pieces:
                    raise gr.Error('No audio generated.')
                full = np.concatenate(pieces)
                return (SAMPLE_RATE, full), "\n".join(log)

            s_lang.change(voice_choices_for, inputs=s_lang, outputs=s_voice)
            s_btn.click(ui_stream, inputs=[s_text, s_voice, s_speed, s_lang], outputs=[s_audio, s_log])

        # ── Tab 3: Tokens (G2P-only) ──
        with gr.Tab("Tokens"):
            gr.Markdown("Inspect how Kokoro tokenizes your text. **No audio is generated** — this is fast and uses no VRAM.")
            with gr.Row():
                with gr.Column():
                    t_lang = gr.Dropdown(choices=LANG_LABELS, value="American English 🇺🇸", label="Language",
                                          info="Picks the G2P backend. Phoneme set varies by language.")
                    t_in = gr.Textbox(lines=4, label="Text",
                                       value="The quick brown fox jumps over the lazy dog.",
                                       info="Markdown-link syntax is honored: [Kokoro](/kˈOkəɹO/)")
                    t_btn = gr.Button("🔤 Tokenize", variant="primary")
                with gr.Column():
                    t_out = gr.Dataframe(headers=["Chunk", "Phonemes"], label="G2P output",
                                          info="Each row is one chunk. Phonemes are Kokoro's IPA-mixed notation.",
                                          wrap=True)
                    t_summary = gr.Textbox(label="Summary", lines=3, interactive=False,
                                            info="Token count, phoneme count, and timing.")

            def ui_tokenize(text, lang_label):
                if not text or not text.strip():
                    raise gr.Error('Text is empty.')
                code = LANG_CODES_BY_LABEL.get(lang_label, 'a')
                t0 = time.time()
                toks = tokenize_only(text, code)
                rows = [[i, gs, ps] for i, (gs, ps) in enumerate(toks)]
                total_phon = sum(len(ps) for _, ps in toks)
                summary = (f"chunks = {len(toks)} · phonemes = {total_phon} · "
                           f"elapsed = {time.time()-t0:.2f}s")
                return rows, summary

            t_btn.click(ui_tokenize, inputs=[t_in, t_lang], outputs=[t_out, t_summary])

        # ── Tab 4: Batch ──
        with gr.Tab("Batch"):
            gr.Markdown("Generate one .wav per line and download them all as a zip.")
            with gr.Row():
                with gr.Column():
                    b_lang = gr.Dropdown(choices=LANG_LABELS, value="American English 🇺🇸", label="Language",
                                          info="Applied to every line in the batch.")
                    b_voice = gr.Dropdown(choices=voice_choices_for("American English 🇺🇸"),
                                           value="af_heart", label="Voice",
                                           info="One voice for the entire batch.")
                    b_speed = gr.Slider(0.5, 2.0, value=1.0, step=0.05, label="Speed",
                                         info="Applied to every line.")
                    b_text = gr.Textbox(lines=10, label="Lines (one per utterance)",
                                         value="The quick brown fox jumps over the lazy dog.\nShe sells seashells by the seashore.\nTo be, or not to be: that is the question.",
                                         info="Blank lines are treated as chunk boundaries.")
                    b_btn = gr.Button("📦 Run batch", variant="primary")
                with gr.Column():
                    b_zip = gr.File(label="Download .zip", info="One WAV per line, named by line index.")
                    b_log = gr.Textbox(label="Per-line log", lines=10, interactive=False,
                                        info="Status of each line — ok / failed / timing.")

            def ui_batch(text, voice, speed, lang_label, progress=gr.Progress()):
                if not text or not text.strip():
                    raise gr.Error('Text is empty.')
                voice = _normalize_voice(voice)
                code = LANG_CODES_BY_LABEL.get(lang_label, 'a')
                lines = [l.strip() for l in text.splitlines() if l.strip()]
                out_dir = '/content/audio_out/batch'
                os.makedirs(out_dir, exist_ok=True)
                log = []
                zip_buf = io.BytesIO()
                with zipfile.ZipFile(zip_buf, 'w', zipfile.ZIP_DEFLATED) as zf:
                    for i, line in enumerate(lines):
                        t0 = time.time()
                        try:
                            chunks = synthesize(line, voice, float(speed), code)
                            if not chunks:
                                raise RuntimeError('no audio chunks')
                            audio, sr = np.concatenate([c[2] for c in chunks]), SAMPLE_RATE
                            fname = f'line_{i:03d}.wav'
                            wav_path = save_wav(audio, sr, fname)
                            # Add to zip
                            with open(wav_path, 'rb') as f:
                                zf.writestr(fname, f.read())
                            log.append(f"[{i:03d}] ok · {len(audio)/sr:.2f}s · {time.time()-t0:.1f}s · {line[:40]}{'…' if len(line)>40 else ''}")
                        except Exception as e:
                            log.append(f"[{i:03d}] FAILED · {e}")
                        progress((i + 1) / max(len(lines), 1), f"{i+1}/{len(lines)}")
                zip_buf.seek(0)
                zip_path = '/content/audio_out/batch.zip'
                with open(zip_path, 'wb') as f:
                    f.write(zip_buf.read())
                return zip_path, "\n".join(log)

            b_lang.change(voice_choices_for, inputs=b_lang, outputs=b_voice)
            b_btn.click(ui_batch, inputs=[b_text, b_voice, b_speed, b_lang], outputs=[b_zip, b_log])

        # ── Tab 5: VRAM ──
        with gr.Tab("VRAM"):
            gr.Markdown("Kokoro is small (~330 MB). This tab is mostly informational. Loaded pipelines are listed below; click **Free all** to release them.")
            v_log = gr.Textbox(label="Loaded pipelines", lines=4, interactive=False,
                                info="Format: lang_code → device.")
            v_btn = gr.Button("🧹 Free all pipelines", variant="secondary")

            def ui_vram_list():
                if not PIPELINES:
                    return "(no pipelines loaded yet)"
                return "\n".join(f"{k!r}  →  loaded" for k in PIPELINES.keys())

            def ui_free_all():
                free_pipeline(None)
                if torch.cuda.is_available():
                    free = torch.cuda.mem_get_info(0)[0] / 1e9
                    tot = torch.cuda.mem_get_info(0)[1] / 1e9
                    return f"Freed all. GPU memory: {free:.1f} / {tot:.1f} GB free."
                return "Freed all. (CPU mode — no GPU to query.)"

            v_btn.click(ui_free_all, outputs=v_log)
            demo.load(ui_vram_list, outputs=v_log)

        # ── Tab 6: Help ──
        with gr.Tab("Help"):
            gr.Markdown(
                """## Custom pronunciation

Use Markdown-link syntax inside the textbox:

```
[Kokoro](/kˈOkəɹO/) is an open-weight TTS model.
```

The `Kokoro` text is replaced with the IPA inside the parentheses. Anything after the closing `)` is the rest of the grapheme sequence.

## Voice blending

Pass a comma-separated list of voice ids to average two voices:

```
af_bella,am_michael
```

This gives a neutral androgynous voice in between. Order doesn't matter.

## Language codes

| Code | Language | Notes |
| --- | --- | --- |
""" +
                "\n".join(f"| `{c}` | {lbl} | {note} |" for c, lbl, note in LANG_TABLE) +
                "\n\n## Voice codes (54 total)\n\n" +
                "\n".join(f"- `{v[0]}` — {v[2]} · {v[3]} · quality {v[4]} · duration {v[5]} · grade {v[6]}"
                           for v in VOICES) +
                "\n\nSee [VOICES.md](https://huggingface.co/hexgrad/Kokoro-82M/blob/main/VOICES.md) for full quality/duration grades.\n\n" +
                "## Citation\n\n```bibtex\n@inproceedings{kokoro-82m,\n  title={{Kokoro-82M}: An Open-Weight 82M-Param TTS Model},\n  author={{hexgrad}},\n  year={{2025}},\n  url={{https://huggingface.co/hexgrad/Kokoro-82M}}\n}\n```"
            )

# ── Welcome message on first load ──
def _welcome():
    n = len(VOICES)
    return f"🎙️ Kokoro-82M ready. {n} voices across {len(VOICES_BY_LANG)} languages. CPU-friendly; GPU is optional."

demo.load(_welcome, outputs=[])

demo.queue(max_size=8, concurrency_limit=2).launch(
    share=True, show_error=True, server_name="0.0.0.0", server_port=7860,
)


In [ ]:
# @title Step 5 — Keep Alive
# @markdown Prevents Colab from disconnecting during long generation sessions. Run anytime after launching the UI.

import threading, time
def _keep():
    while True:
        time.sleep(60)
        try:
            import requests
            requests.get('https://www.google.com', timeout=5)
        except Exception:
            pass
threading.Thread(target=_keep, daemon=True).start()
print('[Keep-Alive] Running. Stop cell to disable.')


In [ ]:
# @title Step 6 — Quick Test (one-off inference, no UI)
# @markdown Run a single TTS call from the notebook. Useful for scripting or quick checks.

LANG_CODE = "a"  # @param ["a", "b", "e", "f", "h", "i", "p", "j", "z"] {type:"string"}
VOICE = "af_heart"  # @param {type:"string"}
SPEED = 1.0  # @param {type:"slider", min:0.5, max:2.0, step:0.05}
TEXT = "[Kokoro](/kˈOkəɹO/) is an open-weight TTS model with 82 million parameters."  # @param {type:"string"}
DEVICE_OVERRIDE = "auto"  # @param ["auto", "cuda", "cpu"]

import time, os
device = None if DEVICE_OVERRIDE == "auto" else DEVICE_OVERRIDE
print(f'[Test] lang={LANG_CODE!r} voice={VOICE!r} speed={SPEED:.2f} device={device or "auto"}')
print(f'[Test] text = {TEXT!r}')

t0 = time.time()
audio, sr = synthesize_concat(TEXT, voice=VOICE, speed=SPEED, lang_code=LANG_CODE, device=device)
elapsed = time.time() - t0
path = save_wav(audio, sr, f'quicktest_{int(time.time())}.wav')
print(f'\n[Done] {path}')
print(f'[Done] {len(audio)} samples · {len(audio)/sr:.2f}s @ {sr} Hz · generated in {elapsed:.2f}s')

from IPython.display import Audio, display
display(Audio(path, autoplay=False))


In [ ]:
# @title Step 7 — Batch Synthesis
# @markdown Generate audio for a list of texts. Each non-empty line in `BATCH` becomes one output file. Useful for audiobooks, narration, voice prototyping.

BATCH_LANG = "a"  # @param ["a", "b", "e", "f", "h", "i", "p", "j", "z"] {type:"string"}
BATCH_VOICE = "af_heart"  # @param {type:"string"}
BATCH_SPEED = 1.0  # @param {type:"slider", min:0.5, max:2.0, step:0.05}
BATCH_OUT_DIR = "/content/audio_out/batch"  # @param {type:"string"}

BATCH = """\
The quick brown fox jumps over the lazy dog.
She sells seashells by the seashore.
To be, or not to be: that is the question.
It was the best of times, it was the worst of times.
All happy families are alike; each unhappy family is unhappy in its own way.
"""

import os, time
lines = [l.strip() for l in BATCH.strip().splitlines() if l.strip()]
os.makedirs(BATCH_OUT_DIR, exist_ok=True)
print(f'[Batch] {len(lines)} lines · lang={BATCH_LANG!r} voice={BATCH_VOICE!r} speed={BATCH_SPEED:.2f}')

t_start = time.time()
for i, line in enumerate(lines):
    label = f"[{i+1:03d}/{len(lines)}]"
    snippet = line[:60] + ('…' if len(line) > 60 else '')
    print(f'{label} {snippet}', flush=True)
    t0 = time.time()
    try:
        audio, sr = synthesize_concat(line, voice=BATCH_VOICE, speed=BATCH_SPEED, lang_code=BATCH_LANG)
        out_path = os.path.join(BATCH_OUT_DIR, f'{i:03d}.wav')
        sf.write(out_path, audio, sr, subtype='PCM_16')
        print(f'  ✓ {len(audio)/sr:.2f}s · {time.time()-t0:.1f}s · {out_path}', flush=True)
    except Exception as e:
        print(f'  ✗ EXCEPTION: {e}', flush=True)
        continue

elapsed = time.time() - t_start
print(f'\n[Done] {len(lines)} files in {BATCH_OUT_DIR} · total wall time {elapsed:.1f}s')
